# Runpod | REC fine-tune ResNet-101 layer4

Chạy notebook trong thư mục có checkpoint **ResNet-101 REC epoch 10**. Notebook tự tìm file `.pth` hoặc `.pt` trong `PROJECT_DIR`. Nếu cần, sửa `PROJECT_DIR` hoặc điền `WARM_PATH`. Lượt fine-tune bắt đầu ở epoch 11. Checkpoint chỉ chứa fusion/head sẽ dùng lại ResNet-101 pretrained ImageNet và RoBERTa-base pretrained như notebook gốc.

Đặt `DATA_DIR` đến thư mục chứa `person_samples.json` và `images/` nếu notebook chưa tìm thấy dữ liệu. Dùng một GPU Runpod. Mặc định batch và microbatch đều là 128. Khi thiếu VRAM, giảm `MICROBATCH_SIZE` xuống 64, giữ `BATCH_SIZE = 128`.

Cache ảnh lưu đầu ra của stem đến hết `layer3` của ResNet-101. Cache text lưu đầu ra sau đủ 12 layer của RoBERTa. Mỗi ảnh và mỗi câu chỉ trích xuất một lần. Trong lúc train, chạy `layer4` từ cache ảnh rồi đưa feature 2048 kênh cùng cache text vào fusion Transformer và bbox head. Các trọng số `layer4`, fusion và bbox head được cập nhật. BatchNorm của `layer4` dùng thống kê đã lưu.

Lượt đầu cần đủ dung lượng đĩa cho cache. `CACHE_ROOT` và `IMAGE_CACHE_ROOT` có thể trỏ tới ổ lưu trữ khác. Checkpoint fine-tune lưu trong `rec_finetune_resnet101_layer4_bs128_text_frozen/`.


In [1]:
%pip install -q "transformers==4.47.1"


Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -q -U kagglehub

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from getpass import getpass
from pathlib import Path
from shutil import copy2
import kagglehub

os.environ["KAGGLE_API_TOKEN"] = getpass("Dán Kaggle API token: ").strip()

checkpoint = kagglehub.model_download(
    "lamky690/epoch10-ref/pyTorch/default",
    path="epoch10.pth",
)

dest = Path.cwd() / "resnet101_epoch10.pth"
copy2(checkpoint, dest)
print(dest)

Dán Kaggle API token:  ········


/workspace/resnet101_epoch10.pth


In [4]:
WARM_PATH = Path("/workspace/resnet101_epoch10.pth")
print(WARM_PATH.is_file())  # Phải in True

True


In [5]:
import kagglehub

# Download latest version
DATA_DIR = kagglehub.dataset_download("lamky690/refcoco")

print("Path to dataset files:", DATA_DIR)
DATA_DIR = Path("/root/.cache/kagglehub/datasets/lamky690/refcoco/versions/1/refcoco_data")

Path to dataset files: /root/.cache/kagglehub/datasets/lamky690/refcoco/versions/1


In [6]:
!find /workspace -name last_checkpoint.pth

/workspace/rec_finetune_resnet101_layer4_bs128_text_frozen/last_checkpoint.pth
/workspace/rec_finetune_resnet101_layer4_bs128_text_frozen2/last_checkpoint.pth


In [ ]:
from pathlib import Path
import os, json, math, time, random, hashlib, shutil, gc, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet101, ResNet101_Weights
from torchvision.ops import box_convert, generalized_box_iou_loss
from transformers import AutoTokenizer, AutoModel, AutoConfig
from tqdm.auto import tqdm

PROJECT_DIR = Path.cwd()
PROJECT_DIR = Path(PROJECT_DIR).expanduser().resolve()
# DATA_DIR = None  # Ví dụ: Path('/workspace/data/refcoco_data')
WARM_PATH = Path("/workspace/resnet101_epoch10.pth")  # Tự tìm checkpoint frozen epoch 10 trong PROJECT_DIR.
RESUME_PATH = Path("/workspace/rec_finetune_resnet101_layer4_bs128_text_frozen2/last_checkpoint.pth")
OUTPUT_DIR = PROJECT_DIR / 'rec_finetune_resnet101_layer4_bs128_text_frozen2'
STAGE = 'rec_finetune_resnet101_layer4_text_frozen'
IMAGE_BACKBONE = 'resnet101'
IMAGE_WEIGHTS = ResNet101_Weights.DEFAULT
WARM_EPOCH = 9

CACHE_ROOT = PROJECT_DIR / 'cache' / 'roberta_base_layer12'
IMAGE_CACHE_ROOT = PROJECT_DIR / 'cache' / 'resnet101_layer3'
TEXT_MODEL = 'FacebookAI/roberta-base'
TEXT_REVISION = 'main'
IMAGE_DATA_VERSION = 'person_images_v1'  # Đổi nếu thay nội dung JPEG.

SEED = 42
BATCH_SIZE = 64  # Số mẫu mỗi optimizer step.
MICROBATCH_SIZE = 64  # Số mẫu mỗi forward/backward trên GPU.
TEXT_EXTRACT_BATCH_SIZE = 128
IMAGE_EXTRACT_BATCH_SIZE = 32
NUM_WORKERS = 4
PREFETCH_FACTOR = 1
CACHE_DISK_RESERVE_GIB = 6.0
EPOCHS = 50  # Epoch cuối, kể cả khi resume.
MAX_SESSION_HOURS = None
NEXT_EPOCH_RESERVE_MIN = 15.0
L1_WEIGHT, GIOU_WEIGHT = 8.0, 2.0
BACKBONE_LR = 1e-5
OTHER_LR = 1e-5
WEIGHT_DECAY = 1e-4
LOG_EVERY = 25
RUN_TEST = False

assert PROJECT_DIR.is_dir(), f'Sai PROJECT_DIR: {PROJECT_DIR}'
assert 1 <= MICROBATCH_SIZE <= BATCH_SIZE
assert torch.cuda.is_available(), 'Runpod chưa cấp CUDA. Kiểm tra GPU và PyTorch trong pod.'
device = torch.device('cuda:0')

if DATA_DIR is None:
    roots = dict.fromkeys([PROJECT_DIR, Path('/workspace')])
    candidates = [
        root / relative
        for root in roots
        for relative in ('', 'refcoco_data', 'refcoco', 'refcoco/refcoco_data', 'data',
                         'data/refcoco_data', 'data/refcoco', 'data/refcoco/refcoco_data',
                         'datasets/refcoco_data', 'datasets/refcoco/refcoco_data')
    ]
    DATA_DIR = next((path for path in candidates
                     if (path / 'person_samples.json').is_file() and (path / 'images').is_dir()), None)
    if DATA_DIR is None:
        raise FileNotFoundError('Đặt DATA_DIR tới thư mục chứa person_samples.json và images/.')
DATA_DIR = PROJECT_DIR / Path(DATA_DIR).expanduser()
ANNOTATION_PATH = DATA_DIR / 'person_samples.json'
IMAGE_DIR = DATA_DIR / 'images'
assert ANNOTATION_PATH.is_file() and IMAGE_DIR.is_dir(), f'Data thiếu file hoặc thư mục trong: {DATA_DIR}'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(device), '| PyTorch:', str(torch.__version__))
print('Thư mục chạy:', PROJECT_DIR, '| Dữ liệu:', DATA_DIR)
print('Batch:', BATCH_SIZE, '| Microbatch:', MICROBATCH_SIZE)

def loader_workers(batch_bytes):
    """Giới hạn worker để batch lớn không làm đầy /dev/shm của container."""
    shared_memory = Path('/dev/shm')
    if NUM_WORKERS == 0 or not shared_memory.is_dir():
        return 0
    free_bytes = shutil.disk_usage(shared_memory).free
    slots = max(0, free_bytes // batch_bytes - 2)
    return min(NUM_WORKERS, slots // PREFETCH_FACTOR)


In [ ]:
samples = json.loads(ANNOTATION_PATH.read_text(encoding='utf-8'))
train_samples = [s for s in samples if s['split'] == 'train']
val_samples = [s for s in samples if s['split'] == 'val']
test_samples = [s for s in samples if s['split'] == 'testA']
assert train_samples and val_samples and test_samples
unique_train_images = {s['image_id']: s for s in train_samples}
IMAGE_SIZE = (
    round(sum(s['height'] for s in unique_train_images.values()) / len(unique_train_images)),
    round(sum(s['width'] for s in unique_train_images.values()) / len(unique_train_images)),
)
image_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
print('Train / val / testA:', len(train_samples), len(val_samples), len(test_samples))
print('Resize (H, W):', IMAGE_SIZE)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

annotation_sha = file_sha256(ANNOTATION_PATH)
# Kiểm tra cấu trúc dữ liệu trước khi tải model/tạo cache.
for s in samples:
    assert s['width'] > 0 and s['height'] > 0
    assert isinstance(s['text'], str) and s['text'].strip()
    assert len(s['bbox']) == 4 and s['bbox'][2] > 0 and s['bbox'][3] > 0
    assert (IMAGE_DIR / s['file_name']).is_file(), s['file_name']
print('Dữ liệu sẵn sàng cho fine-tune ResNet-101 layer4.')


## Checkpoint ResNet-101 và cache RoBERTa sau 12 layer


In [ ]:
ckpt = torch.load(WARM_PATH, map_location="cpu", weights_only=True)
cfg = ckpt.get("config") or {}
state = ckpt.get("model_state_dict") or {}
proj = state.get("image_projection.weight")
if proj is None:
    proj = state.get("module.image_projection.weight")

checks = {
    "epoch": (ckpt.get("epoch"), WARM_EPOCH),
    "image_projection.shape": (tuple(proj.shape) if proj is not None else None, (256, 2048, 1, 1)),
    "image_backbone": (cfg.get("image_backbone", IMAGE_BACKBONE), IMAGE_BACKBONE),
    "text_model": (cfg.get("text_model", TEXT_MODEL), TEXT_MODEL),
    "image_size_hw": (tuple(cfg.get("image_size_hw") or IMAGE_SIZE), IMAGE_SIZE),
    "annotation_sha256": (cfg.get("annotation_sha256", annotation_sha), annotation_sha),
}

for name, (actual, expected) in checks.items():
    if actual != expected:
        print(f"LỆCH {name} | trong file: {actual} | notebook cần: {expected}")

del ckpt, state, proj

In [ ]:
def find_warm_checkpoint(required_sha=None, previous_name=None):
    if WARM_PATH is None:
        names = [name for name in (
            previous_name, 'last_checkpoint.pth', 'checkpoint_epoch10.pth',
            'epoch10.pth', 'best_checkpoint.pth',
        ) if name]
        candidates = dict.fromkeys([
            *(PROJECT_DIR / name for name in names),
            *sorted(PROJECT_DIR.glob('*.pth')),
            *sorted(PROJECT_DIR.glob('*.pt')),
        ])
    else:
        candidates = [PROJECT_DIR / Path(WARM_PATH).expanduser()]

    for path in candidates:
        if WARM_PATH is None and not path.is_file():
            continue
        try:
            checkpoint = torch.load(path, map_location='cpu', weights_only=True)
        except (OSError, RuntimeError, pickle.UnpicklingError, EOFError):
            if WARM_PATH is not None:
                raise
            continue
        if not isinstance(checkpoint, dict) or checkpoint.get('epoch') != WARM_EPOCH:
            continue
        weights = checkpoint.get('model_state_dict')
        if not isinstance(weights, dict):
            continue
        state = {key.removeprefix('module.'): value for key, value in weights.items()}
        projection = state.get('image_projection.weight')
        if tuple(getattr(projection, 'shape', ())) != (256, 2048, 1, 1):
            continue
        metadata = checkpoint.get('config') or {}
        if (
            not isinstance(metadata, dict)
            or metadata.get('image_backbone', IMAGE_BACKBONE) != IMAGE_BACKBONE
            or metadata.get('text_model', TEXT_MODEL) != TEXT_MODEL
            or metadata.get('annotation_sha256', annotation_sha) != annotation_sha
            or tuple(metadata.get('image_size_hw') or IMAGE_SIZE) != IMAGE_SIZE
        ):
            continue
        sha = file_sha256(path)
        if required_sha is not None and sha != required_sha:
            continue
        return path, state, metadata, sha

    location = PROJECT_DIR if WARM_PATH is None else WARM_PATH
    raise ValueError(
        f'Thiếu checkpoint REC ResNet-101 epoch {WARM_EPOCH} khớp cấu hình tại {location}. '
        'Kiểm tra WARM_PATH và file checkpoint gốc.'
    )

resume = None
warm_state = {}
if RESUME_PATH is None:
    warm_path, warm_state, warm_config, warm_sha = find_warm_checkpoint()
    warm_name = warm_path.name
    TEXT_REVISION = warm_config.get('text_revision') or TEXT_REVISION
    print('Warm-start ResNet-101:', warm_path, '| Epoch:', WARM_EPOCH)
else:
    RESUME_PATH = PROJECT_DIR / Path(RESUME_PATH).expanduser()
    resume = torch.load(RESUME_PATH, map_location='cpu', weights_only=True)
    cfg = resume['config']
    assert cfg['stage'] == STAGE and cfg['image_backbone'] == IMAGE_BACKBONE
    assert cfg['annotation_sha256'] == annotation_sha and tuple(cfg['image_size_hw']) == IMAGE_SIZE
    assert cfg['batch_size'] == BATCH_SIZE
    assert cfg['loss_weights'] == [L1_WEIGHT, GIOU_WEIGHT] and cfg['text_model'] == TEXT_MODEL
    TEXT_REVISION = cfg['text_revision']
    warm_name, warm_sha = cfg['warm_checkpoint_name'], cfg['warm_checkpoint_sha256']
    if cfg['backbones_from_warm']:
        _, warm_state, _, _ = find_warm_checkpoint(required_sha=warm_sha, previous_name=warm_name)

image_weights = {
    key.removeprefix('image_encoder.'): value for key, value in warm_state.items()
    if key.startswith('image_encoder.')
}
text_weights = {
    key.removeprefix('text_encoder.'): value for key, value in warm_state.items()
    if key.startswith('text_encoder.')
}
fusion_weights = {
    key: value for key, value in warm_state.items()
    if not key.startswith(('image_encoder.', 'text_encoder.'))
} if resume is None else {}
del warm_state

backbones_from_warm = bool(image_weights or text_weights)
image_weight_source = f'checkpoint:{warm_sha}' if image_weights else IMAGE_WEIGHTS.url
text_weight_source = f'checkpoint:{warm_sha}' if text_weights else 'pretrained'

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL, revision=TEXT_REVISION)
resolved_revision = tokenizer.init_kwargs.get('_commit_hash')
if not resolved_revision:
    from transformers import AutoConfig
    resolved_revision = AutoConfig.from_pretrained(TEXT_MODEL, revision=TEXT_REVISION)._commit_hash
assert resolved_revision, 'Không xác định được commit RoBERTa để tái lập cache.'

all_samples = train_samples + val_samples + test_samples
unique_texts = sorted({s['text'] for s in all_samples})
text_to_index = {t:i for i,t in enumerate(unique_texts)}
text_hash = hashlib.sha256(json.dumps(unique_texts, ensure_ascii=False).encode()).hexdigest()
cache_spec = dict(
    text_model=TEXT_MODEL, revision=resolved_revision, weight_source=text_weight_source,
    texts_sha256=text_hash, annotation_sha256=annotation_sha,
    dtype='float16', output='after_layer_12',
)
cache_id = hashlib.sha256(json.dumps(cache_spec, sort_keys=True).encode()).hexdigest()[:20]
CACHE_DIR = CACHE_ROOT / cache_id
manifest_path = CACHE_DIR / 'metadata.json'

if manifest_path.is_file():
    cache_meta = json.loads(manifest_path.read_text())
    assert cache_meta['spec'] == cache_spec
    assert (CACHE_DIR / 'features.npy').is_file()
    print('Tái sử dụng text cache:', CACHE_DIR)
else:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    encoded = tokenizer(unique_texts, padding=False, truncation=False)
    lengths = np.array([len(x) for x in encoded['input_ids']], dtype=np.int64)
    assert lengths.min() > 0
    assert int(lengths.max()) <= tokenizer.model_max_length
    offsets = np.concatenate([np.zeros(1, dtype=np.int64), np.cumsum(lengths)])
    required_bytes = int(offsets[-1]) * 768 * 2
    assert shutil.disk_usage(CACHE_DIR).free > required_bytes + 2 * 1024**3, 'Thiếu đĩa cho cache.'

    if text_weights:
        text_config = AutoConfig.from_pretrained(TEXT_MODEL, revision=resolved_revision)
        text_encoder = AutoModel.from_config(text_config, add_pooling_layer=False)
        text_encoder.load_state_dict(text_weights, strict=True)
    else:
        text_encoder = AutoModel.from_pretrained(
            TEXT_MODEL, revision=resolved_revision, add_pooling_layer=False,
        )
    text_encoder.requires_grad_(False)
    text_encoder = text_encoder.to(device).eval()
    cache = np.lib.format.open_memmap(
        CACHE_DIR / 'features.npy', mode='w+', dtype=np.float16,
        shape=(int(offsets[-1]), 768),
    )
    with torch.no_grad():
        for start in tqdm(range(0, len(unique_texts), TEXT_EXTRACT_BATCH_SIZE), desc='Cache RoBERTa'):
            end = min(start + TEXT_EXTRACT_BATCH_SIZE, len(unique_texts))
            inputs = tokenizer.pad(
                {k:v[start:end] for k,v in encoded.items()},
                padding=True, return_tensors='pt',
            ).to(device)
            with torch.autocast('cuda', dtype=torch.float16):
                features = text_encoder(**inputs).last_hidden_state
            features = features.cpu().to(torch.float16).numpy()
            for j, i in enumerate(range(start, end)):
                cache[offsets[i]:offsets[i+1]] = features[j, :lengths[i]]
    cache.flush()
    np.save(CACHE_DIR / 'offsets.npy', offsets)
    cache_meta = dict(spec=cache_spec, rows=int(offsets[-1]), hidden_size=768,
                      max_length=int(lengths.max()), unique_texts=len(unique_texts))
    # Chỉ tạo manifest khi cache hoàn chỉnh.
    temp_manifest = manifest_path.with_suffix('.tmp')
    temp_manifest.write_text(json.dumps(cache_meta, ensure_ascii=False, indent=2))
    os.replace(temp_manifest, manifest_path)
    del cache, encoded, features, inputs, text_encoder, lengths, offsets
    gc.collect()
    torch.cuda.empty_cache()
text_check = np.load(CACHE_DIR / 'features.npy', mmap_mode='r')
text_offsets = np.load(CACHE_DIR / 'offsets.npy', mmap_mode='r')
assert text_check.dtype == np.float16 and text_check.shape == (cache_meta['rows'], 768)
assert len(text_offsets) == len(unique_texts)+1 and text_offsets[-1] == len(text_check)
del text_check, text_offsets, text_weights
print('Cache:', cache_meta)


## Cache ResNet-101 từ stem tới hết layer3


In [ ]:
# Resume từ pretrained dùng layer4 đã lưu. Chỉ tải ImageNet nếu cache ảnh thiếu.
image_encoder = None
if image_weights or resume is None:
    backbone = resnet101(weights=None if image_weights else IMAGE_WEIGHTS)
    image_encoder = nn.Sequential(*list(backbone.children())[:-2])
    del backbone
    if image_weights:
        image_encoder.load_state_dict(image_weights, strict=True)
del image_weights


In [ ]:
unique_images = sorted({s['image_id']:s for s in all_samples}.values(), key=lambda s:s['image_id'])
image_to_index = {str(s['image_id']):i for i,s in enumerate(unique_images)}
image_list_sha = hashlib.sha256(json.dumps([
    [s['image_id'],s['file_name'],s['width'],s['height']] for s in unique_images
],ensure_ascii=False).encode()).hexdigest()
image_cache_spec = dict(
    model=IMAGE_BACKBONE, weight_source=image_weight_source, output='resnet101_layer3',
    images_sha256=image_list_sha, image_data_version=IMAGE_DATA_VERSION,
    annotation_sha256=annotation_sha, image_size_hw=list(IMAGE_SIZE),
    mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225],
    dtype='float16', input_dtype='float32_normalized', stride=16,
    torch_version=str(torch.__version__), torchvision_version=str(__import__('torchvision').__version__),
)
image_cache_id = hashlib.sha256(json.dumps(image_cache_spec,sort_keys=True).encode()).hexdigest()[:20]
IMAGE_CACHE_DIR = IMAGE_CACHE_ROOT / image_cache_id
image_meta_path = IMAGE_CACHE_DIR / 'metadata.json'
image_feature_path = IMAGE_CACHE_DIR / 'features.npy'

class ExtractionImages(Dataset):
    def __len__(self): return len(unique_images)
    def __getitem__(self,index):
        sample = unique_images[index]
        with Image.open(IMAGE_DIR / sample['file_name']) as im:
            im = im.convert('RGB')
            assert im.size == (sample['width'],sample['height']), 'Metadata ảnh sai.'
            return image_transform(im)

if image_meta_path.is_file():
    image_meta = json.loads(image_meta_path.read_text())
    assert image_meta['spec'] == image_cache_spec
    print('Tái sử dụng cache ResNet layer3:', IMAGE_CACHE_DIR)
else:
    IMAGE_CACHE_DIR.mkdir(parents=True,exist_ok=True)
    feature_shape = [1024, math.ceil(IMAGE_SIZE[0]/16), math.ceil(IMAGE_SIZE[1]/16)]
    required_bytes = len(unique_images)*math.prod(feature_shape)*2
    reusable = image_feature_path.stat().st_size if image_feature_path.exists() else 0
    available = shutil.disk_usage(IMAGE_CACHE_DIR).free + reusable
    if available <= required_bytes * 1.02 + CACHE_DISK_RESERVE_GIB * 1024**3:
        raise OSError(
            f'Cache ảnh cần khoảng {required_bytes / 1024**3:.1f} GiB '
            f'và {CACHE_DISK_RESERVE_GIB:.1f} GiB dự phòng. Đổi IMAGE_CACHE_ROOT sang ổ còn chỗ.'
        )
    print('Feature/ảnh:', feature_shape, '| Cache GiB:', required_bytes / 1024**3)
    if image_encoder is None:
        backbone = resnet101(weights=IMAGE_WEIGHTS)
        image_encoder = nn.Sequential(*list(backbone.children())[:-2])
        del backbone
    frozen_image = image_encoder[:7].requires_grad_(False).eval().to(device)
    extract_batch_bytes = IMAGE_EXTRACT_BATCH_SIZE * 3 * math.prod(IMAGE_SIZE) * 4
    extract_workers = loader_workers(extract_batch_bytes)
    options = dict(batch_size=IMAGE_EXTRACT_BATCH_SIZE, shuffle=False,
                   num_workers=extract_workers, pin_memory=True)
    if extract_workers > 0:
        options['prefetch_factor'] = PREFETCH_FACTOR
    extract_loader = DataLoader(ExtractionImages(), **options)
    cache = np.lib.format.open_memmap(image_feature_path, mode='w+',dtype=np.float16,
                                     shape=(len(unique_images),*feature_shape))
    offset = 0
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for images in tqdm(extract_loader,desc='Cache ResNet layer3 / ảnh duy nhất'):
            features = frozen_image(images.to(device, non_blocking=True))
            assert list(features.shape[1:]) == feature_shape
            assert torch.isfinite(features).all().item(), 'ResNet-101 layer3 có NaN/Inf.'
            features = features.to(torch.float16).cpu().numpy()
            cache[offset:offset+len(features)] = features
            offset += len(features)
    assert offset == len(unique_images)
    cache.flush()
    image_meta = dict(spec=image_cache_spec, feature_shape=feature_shape, num_images=len(unique_images))
    temp_manifest = image_meta_path.with_suffix('.tmp')
    temp_manifest.write_text(json.dumps(image_meta, ensure_ascii=False, indent=2))
    os.replace(temp_manifest, image_meta_path)
    del cache, features, images, frozen_image, extract_loader
    gc.collect()
    torch.cuda.empty_cache()
check = np.load(image_feature_path,mmap_mode='r')
assert check.dtype == np.float16
assert list(check.shape) == [len(unique_images)] + image_meta['feature_shape']
assert image_meta['feature_shape'][0] == 1024
del check
if resume is not None:
    assert resume['config']['image_cache_spec'] == image_cache_spec, 'Cache/config đã đổi.'
    assert resume['config']['text_cache_spec'] == cache_spec


## Fine-tune layer4 và fusion/head của REC


In [13]:
class PositionEncoding2D(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        assert d_model % 4 == 0
        self.d_model = d_model

    def forward(self, feature_map):
        _, _, H, W = feature_map.shape
        device = feature_map.device
        n = self.d_model // 4

        frequencies = 10000 ** (
            torch.arange(n, device=device, dtype=torch.float32) / n
        )

        y = torch.linspace(0, 2 * math.pi, H, device=device)
        x = torch.linspace(0, 2 * math.pi, W, device=device)

        y = y[:, None] / frequencies[None, :]
        x = x[:, None] / frequencies[None, :]

        pos_y = torch.cat([y.sin(), y.cos()], dim=-1)
        pos_x = torch.cat([x.sin(), x.cos()], dim=-1)

        pos_y = pos_y[:, None, :].expand(H, W, -1)
        pos_x = pos_x[None, :, :].expand(H, W, -1)

        position = torch.cat([pos_y, pos_x], dim=-1)

        return position.reshape(
            1, H * W, self.d_model
        ).to(dtype=feature_map.dtype)


class FineTuneREC(nn.Module):
    def __init__(self, layer4, d_model=256):
        super().__init__()
        self.layer4 = layer4.requires_grad_(True).eval()
        self.image_projection = nn.Conv2d(2048, d_model, kernel_size=1)
        self.image_position = PositionEncoding2D(d_model)
        self.text_projection = nn.Linear(768, d_model)
        self.modality_embedding = nn.Embedding(2, d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=8, num_encoder_layers=6,
            num_decoder_layers=6, dim_feedforward=1024,
            dropout=0.1, activation='relu', batch_first=True,
        )
        self.object_query = nn.Embedding(1, d_model)
        self.box_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.ReLU(),
            nn.Linear(d_model, d_model), nn.ReLU(),
            nn.Linear(d_model, 4), nn.Sigmoid(),
        )

    def train(self, mode=True):
        super().train(mode)
        # BatchNorm giữ running stats, các tham số layer4 vẫn nhận gradient.
        self.layer4.eval()
        return self

    @torch.autocast('cuda', dtype=torch.float16)
    def forward(self, image_features, text_features, attention_mask):
        x = self.image_projection(self.layer4(image_features))
        image_tokens = x.flatten(2).transpose(1, 2)
        image_tokens = image_tokens + self.image_position(x) + self.modality_embedding.weight[0]
        text_tokens = self.text_projection(text_features) + self.modality_embedding.weight[1]
        tokens = torch.cat([image_tokens, text_tokens], dim=1)
        image_padding = torch.zeros(x.shape[0], image_tokens.shape[1], dtype=torch.bool, device=x.device)
        padding = torch.cat([image_padding, ~attention_mask.bool()], dim=1)
        memory = self.transformer.encoder(tokens, src_key_padding_mask=padding)
        query = self.object_query.weight.unsqueeze(0).expand(x.shape[0], -1, -1)
        decoded = self.transformer.decoder(query, memory, memory_key_padding_mask=padding)
        return self.box_head(decoded[:, 0, :])


layer4 = resnet101(weights=None).layer4 if image_encoder is None else image_encoder[7]
base_model = FineTuneREC(layer4)
if resume is not None:
    base_model.load_state_dict(resume['model_state_dict'], strict=True)
else:
    expected = {key for key in base_model.state_dict() if not key.startswith('layer4.')}
    if fusion_weights.keys() != expected:
        missing = sorted(expected - fusion_weights.keys())
        unexpected = sorted(fusion_weights.keys() - expected)
        raise ValueError(f'Checkpoint fusion/head không khớp. Thiếu: {missing}. Dư: {unexpected}.')
    base_model.load_state_dict(fusion_weights, strict=False)
del fusion_weights, image_encoder, layer4
gc.collect()
torch.cuda.empty_cache()
base_model = base_model.to(device)
model = base_model


In [14]:
class CachedRECDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
        self.features = self.offsets = self.image_features = None
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        if self.features is None:
            self.features = np.load(CACHE_DIR / 'features.npy', mmap_mode='r')
            self.offsets = np.load(CACHE_DIR / 'offsets.npy', mmap_mode='r')
            self.image_features = np.load(IMAGE_CACHE_DIR / 'features.npy', mmap_mode='r')
        s = self.rows[index]
        image = torch.from_numpy(self.image_features[image_to_index[str(s['image_id'])]].copy())
        W, H = s['width'], s['height']
        x,y,w,h = s['bbox']
        box = torch.tensor([(x+w/2)/W, (y+h/2)/H, w/W, h/H], dtype=torch.float32)
        idx = text_to_index[s['text']]
        text = torch.from_numpy(self.features[self.offsets[idx]:self.offsets[idx+1]].copy())
        return image, text, box

def collate(batch):
    images, texts, boxes = zip(*batch)
    lengths = torch.tensor([len(t) for t in texts])
    text = nn.utils.rnn.pad_sequence(texts, batch_first=True)
    mask = torch.arange(text.shape[1])[None,:] < lengths[:,None]
    return dict(image_features=torch.stack(images), text_features=text,
                attention_mask=mask, boxes=torch.stack(boxes))

train_generator = torch.Generator().manual_seed(SEED)
def worker_init(worker_id):
    seed = torch.initial_seed() % (2**32)
    random.seed(seed)
    np.random.seed(seed)

image_bytes = math.prod(image_meta['feature_shape']) * 2
text_bytes = cache_meta['max_length'] * (cache_meta['hidden_size'] * 2 + 1)
loader_batch_bytes = BATCH_SIZE * (image_bytes + text_bytes + 4 * 4)
train_workers = loader_workers(loader_batch_bytes)

def make_loader(rows, training):
    options = dict(batch_size=BATCH_SIZE, shuffle=training,
                   num_workers=train_workers, pin_memory=True,
                   collate_fn=collate, worker_init_fn=worker_init,
                   persistent_workers=False)
    # Không giữ nhiều bộ worker train/val/test đồng thời trong RAM.
    if train_workers > 0:
        options['prefetch_factor'] = PREFETCH_FACTOR
    if training:
        options['generator'] = train_generator
    return DataLoader(CachedRECDataset(rows), **options)

train_loader = make_loader(train_samples, True)
val_loader = make_loader(val_samples, False)
test_loader = make_loader(test_samples, False)
print('Train batches:', len(train_loader), '| Val batches:', len(val_loader), '| Loader workers:', train_workers)


Train batches: 944 | Val batches: 86 | Loader workers: 4


In [15]:
def grounding_loss(pred, target):
    pred, target = pred.float(), target.float()
    l1 = F.l1_loss(pred, target, reduction='none').sum(-1).mean()
    p = box_convert(pred, in_fmt='cxcywh', out_fmt='xyxy')
    t = box_convert(target, in_fmt='cxcywh', out_fmt='xyxy')
    return L1_WEIGHT*l1 + GIOU_WEIGHT*generalized_box_iou_loss(p,t,reduction='mean')

@torch.no_grad()
def matched_iou(pred, target):
    # IoU từng cặp, không tạo ma trận B×B rồi lấy đường chéo.
    p = box_convert(pred.float(), in_fmt='cxcywh', out_fmt='xyxy')
    t = box_convert(target.float(), in_fmt='cxcywh', out_fmt='xyxy')
    wh = (torch.minimum(p[:,2:],t[:,2:])-torch.maximum(p[:,:2],t[:,:2])).clamp(min=0)
    intersection = wh.prod(-1)
    union = (p[:,2:]-p[:,:2]).clamp(min=0).prod(-1) + (t[:,2:]-t[:,:2]).clamp(min=0).prod(-1) - intersection
    return intersection / union.clamp(min=1e-7)

resnet_params = list(base_model.layer4.parameters())
fusion_params = [
    p for name, p in base_model.named_parameters()
    if p.requires_grad and not name.startswith('layer4.')
]
trainable_params = resnet_params + fusion_params
optimizer = torch.optim.AdamW([
    dict(params=resnet_params, lr=BACKBONE_LR, name='resnet_layer4'),
    dict(params=fusion_params, lr=OTHER_LR, name='fusion_head'),
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
scaler = torch.amp.GradScaler('cuda')

if resume is not None:
    optimizer.load_state_dict(resume['optimizer_state_dict'])
    scheduler.load_state_dict(resume['scheduler_state_dict'])
    scaler.load_state_dict(resume['scaler_state_dict'])
    train_generator.set_state(resume['train_generator_state'])
    random.setstate(resume['python_rng_state'])
    ns = resume['numpy_rng_state']
    np.random.set_state((ns[0], np.array(ns[1], dtype=np.uint32), ns[2], ns[3], ns[4]))
    torch.set_rng_state(resume['torch_rng_state'])
    torch.cuda.set_rng_state(resume['cuda_rng_states'][0], device=device)

def run_epoch(loader, training, description):
    model.train(training)
    # Cộng dồn trên GPU; chỉ lấy số liệu và kiểm tra loss khi log/cuối epoch.
    totals = torch.zeros(4, dtype=torch.float64, device=device)
    seen = 0
    checked_grad = False
    torch.cuda.synchronize()
    started = time.perf_counter()
    progress = tqdm(loader, desc=description, mininterval=2.0)
    for step, batch in enumerate(progress, 1):
        batch_size = len(batch['boxes'])
        if training:
            optimizer.zero_grad(set_to_none=True)
        for start in range(0, batch_size, MICROBATCH_SIZE):
            stop = min(start + MICROBATCH_SIZE, batch_size)
            microbatch = {
                key: value[start:stop].to(device, non_blocking=True)
                for key, value in batch.items()
            }
            n = stop - start
            with torch.set_grad_enabled(training):
                pred = model(microbatch['image_features'], microbatch['text_features'], microbatch['attention_mask'])
                loss = grounding_loss(pred, microbatch['boxes'])
                if training:
                    # Trọng số theo số mẫu thực tế, kể cả batch cuối ngắn hơn 128.
                    scaler.scale(loss * (n / batch_size)).backward()
            iou = matched_iou(pred.detach(), microbatch['boxes'])
            totals += torch.stack([
                loss.detach().double() * n,
                iou.double().sum(),
                (iou >= 0.5).double().sum(),
                (~torch.isfinite(loss.detach())).double(),
            ])
        if training:
            scaler.unscale_(optimizer)
            if not checked_grad:
                assert any(p.grad is not None for p in resnet_params), 'ResNet layer4 thiếu gradient!'
                assert base_model.image_projection.weight.grad is not None, 'Projection thiếu gradient!'
                assert base_model.box_head[0].weight.grad is not None, 'Head không có gradient!'
                checked_grad = True
            nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
        seen += batch_size
        if step % LOG_EVERY == 0 or step == len(loader):
            values = (totals/seen).tolist()
            if values[3] > 0 or not all(math.isfinite(v) for v in values[:3]):
                raise RuntimeError('Có loss/metric NaN hoặc Inf từ lần log trước. Dừng, không ghi đè checkpoint tốt. Nạp lại last/best trước khi thử tiếp.')
            elapsed = time.perf_counter()-started
            progress.set_postfix(loss=f'{values[0]:.4f}', mIoU=f'{values[1]:.4f}', acc50=f'{values[2]:.4f}', samples_s=f'{seen/elapsed:.1f}')
    assert seen > 0
    torch.cuda.synchronize()
    elapsed = time.perf_counter()-started
    values = (totals/seen).tolist()
    if values[3] > 0 or not all(math.isfinite(v) for v in values[:3]):
        raise RuntimeError('Loss/metric NaN hoặc Inf ở cuối epoch; không lưu checkpoint.')
    return dict(loss=values[0], miou=values[1], acc50=values[2], minutes=elapsed/60, samples_s=seen/elapsed)


## Huấn luyện và resume

Mặc định `EPOCHS = 60` là epoch cuối, bắt đầu fine-tune ở epoch 11 khi dùng `WARM_PATH`. Notebook chọn best checkpoint theo validation Acc@0.5. Có thể đặt `MAX_SESSION_HOURS` để kết thúc sau một epoch khi sắp hết thời gian phiên.

Khi resume, gắn `last_checkpoint.pth` và `best_checkpoint.pth` của bản này vào cùng thư mục rồi trỏ `RESUME_PATH` tới file last. Nếu checkpoint gốc chứa trọng số backbone, giữ file epoch 10 trong `PROJECT_DIR` hoặc đặt `WARM_PATH` để có thể tạo lại cache.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = OUTPUT_DIR / 'best_checkpoint.pth'
LAST_PATH = OUTPUT_DIR / 'last_checkpoint.pth'
CONFIG = dict(
    stage=STAGE, annotation_sha256=annotation_sha,
    text_model=TEXT_MODEL, text_revision=resolved_revision, text_cache_spec=cache_spec,
    image_backbone=IMAGE_BACKBONE, warm_checkpoint_name=warm_name, warm_checkpoint_sha256=warm_sha,
    backbones_from_warm=backbones_from_warm,
    image_size_hw=list(IMAGE_SIZE), batch_size=BATCH_SIZE, microbatch_size=MICROBATCH_SIZE,
    image_cache_spec=image_cache_spec, image_feature_shape=image_meta['feature_shape'],
    loss_weights=[L1_WEIGHT,GIOU_WEIGHT], initial_layer4_lr=BACKBONE_LR, initial_other_lr=OTHER_LR,
    trainable_image_layers=['layer4'], trainable_text_layers=[],
    d_model=256, encoder_layers=6, decoder_layers=6, nhead=8,
    dim_feedforward=1024, dropout=0.1, activation='relu', object_queries=1,
    num_workers=NUM_WORKERS, prefetch_factor=PREFETCH_FACTOR,
    seed=SEED, cudnn_benchmark=True, torch_version=str(torch.__version__),
)

def save_checkpoint(path, epoch):
    ns = np.random.get_state()
    payload = dict(
        epoch=epoch, model_state_dict=base_model.state_dict(),
        optimizer_state_dict=optimizer.state_dict(), scheduler_state_dict=scheduler.state_dict(),
        scaler_state_dict=scaler.state_dict(), baseline_metrics=baseline_metrics,
        best_acc50=best_acc50, best_epoch=best_epoch, history=history, config=CONFIG,
        train_generator_state=train_generator.get_state(),
        python_rng_state=random.getstate(),
        numpy_rng_state=[ns[0],ns[1].tolist(),ns[2],ns[3],ns[4]],
        torch_rng_state=torch.get_rng_state(), cuda_rng_states=[torch.cuda.get_rng_state(device)],
    )
    temp = path.with_suffix('.tmp')
    torch.save(payload, temp)
    os.replace(temp, path)

session_started = time.perf_counter()
if resume is None:
    if BEST_PATH.exists() or LAST_PATH.exists():
        raise FileExistsError('Output đã có checkpoint. Đổi OUTPUT_DIR hoặc đặt RESUME_PATH đúng; không ghi đè lượt cũ.')
    baseline_metrics = run_epoch(val_loader, False, 'Validation lúc epoch 10')
    print('Validation checkpoint epoch 10:', baseline_metrics)
    best_acc50, best_epoch = baseline_metrics['acc50'], 10
    history, start_epoch = [], 11
    save_checkpoint(BEST_PATH, 10)
    save_checkpoint(LAST_PATH, 10)
else:
    baseline_metrics = resume['baseline_metrics']
    history = resume['history']
    best_acc50, best_epoch = resume['best_acc50'], resume['best_epoch']
    start_epoch = resume['epoch'] + 1
    source_best = RESUME_PATH.parent / 'best_checkpoint.pth'
    assert source_best.is_file(), 'Resume cần best_checkpoint.pth bên cạnh last để giữ best toàn bộ quá trình.'
    best_check = torch.load(source_best,map_location='cpu',weights_only=True)
    best_config = {key: value for key, value in best_check['config'].items() if key != 'microbatch_size'}
    resume_config = {key: value for key, value in resume['config'].items() if key != 'microbatch_size'}
    assert best_config == resume_config, 'Best và last khác cấu hình.'
    assert best_check['epoch'] == best_epoch and best_check['best_acc50'] == best_acc50
    del best_check
    if source_best.resolve() != BEST_PATH.resolve():
        shutil.copy2(source_best, BEST_PATH)
    del resume
    gc.collect()
    print('Tiếp tục từ epoch', start_epoch, '| Best epoch:', best_epoch)

for epoch in range(start_epoch, EPOCHS+1):
    epoch_started = time.perf_counter()
    lrs = {g['name']:g['lr'] for g in optimizer.param_groups}
    print(f'\nEpoch {epoch}/{EPOCHS}, LR={lrs}')
    train_metrics = run_epoch(train_loader, True, f'Epoch {epoch} | Train REC / fine-tune ResNet-101 layer4')
    val_metrics = run_epoch(val_loader, False, f'Epoch {epoch} | Validation')
    scheduler.step()
    history.append(dict(epoch=epoch, train=train_metrics, val=val_metrics, learning_rates_used=lrs))
    improved = val_metrics['acc50'] > best_acc50
    if improved:
        best_acc50, best_epoch = val_metrics['acc50'], epoch
        save_checkpoint(BEST_PATH, epoch)
    save_checkpoint(LAST_PATH, epoch)
    (OUTPUT_DIR/'history.json').write_text(json.dumps(dict(
        baseline=baseline_metrics, best_epoch=best_epoch, best_acc50=best_acc50,
        config=CONFIG, history=history), ensure_ascii=False, indent=2))
    epoch_seconds = time.perf_counter()-epoch_started
    print('Train:', train_metrics, '\nVal:', val_metrics)
    print(f'Best Acc@0.5={best_acc50:.4f} ở epoch {best_epoch}; cả epoch + lưu: {epoch_seconds/60:.1f} phút')
    if MAX_SESSION_HOURS is not None:
        remaining = MAX_SESSION_HOURS*3600 - (time.perf_counter()-session_started)
        if remaining < epoch_seconds + NEXT_EPOCH_RESERVE_MIN*60:
            print('Kết thúc lượt này để giữ output. Tiếp tục từ last_checkpoint.pth ở phiên sau.')
            break

print('Best:', best_acc50, '| Epoch:', best_epoch)
print('Tăng so epoch 10:', round((best_acc50-baseline_metrics['acc50'])*100,3), 'điểm phần trăm')
print('Output:', OUTPUT_DIR)


Tiếp tục từ epoch 48 | Best epoch: 30

Epoch 48/50, LR={'resnet_layer4': 1.25e-06, 'fusion_head': 1.25e-06}


Epoch 48 | Train REC / fine-tune ResNet-101 layer4:   0%|          | 0/944 [00:00<?, ?it/s]

In [ ]:
# Không dùng testA để điều chỉnh cấu hình hoặc chọn epoch.
if RUN_TEST:
    selected = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
    base_model.load_state_dict(selected['model_state_dict'], strict=True)
    del selected
    test_metrics = run_epoch(test_loader, False, 'TestA | best checkpoint')
    (OUTPUT_DIR/'test_metrics.json').write_text(json.dumps(test_metrics, indent=2))
    print(test_metrics)
else:
    print('Bỏ qua testA. Bật RUN_TEST sau khi chốt thí nghiệm.')
